In [5]:
import os
import torch
import numpy as np
os.environ["KERAS_BACKEND"] = "torch"

In [6]:
if torch.backends.mps.is_available():
    print("Apple's MPS backend (used by PyTorch on M1/M2 Macs) does not support float64 (double precision).")
    mps_enabled = True
    torch.set_default_dtype(torch.float32)
    print("set default to float32")

Apple's MPS backend (used by PyTorch on M1/M2 Macs) does not support float64 (double precision).
set default to float32


In [7]:
import keras
from utils.project_utils import update_json, standard_plot, plot_confusion_matrix
from executors.executors import build_model_single_hidden, build_model_two_hidden
from utils.data_loader import get_monk_data

## Part 1: Monk

In [ ]:
# --- MONK GRID SEARCH (SINGLE SPLIT) ---
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt

# Global Plot Settings
plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

batch_size = 256


# --- HELPER: Build Monk Model ---
def build_monk_model(hidden_units, learning_rate, momentum, l2_reg, input_size):
    reg = keras.regularizers.l2(l2_reg) if l2_reg > 0 else None
    model = keras.Sequential([
        keras.layers.Dense(hidden_units, activation='relu', input_shape=(input_size,), kernel_regularizer=reg),
        keras.layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=keras.optimizers.SGD(learning_rate=learning_rate, momentum=momentum),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

experiments = [
    #{'id': 1, 'reg': False, 'name': 'Monk-1'},
    #{'id': 2, 'reg': False, 'name': 'Monk-2'},
    #{'id': 3, 'reg': False, 'name': 'Monk-3'},
    {'id': 3, 'reg': True, 'name': 'Monk-3-Reg'}
]

for exp in experiments:
    monk_id = exp['id']
    is_reg = exp['reg']
    task_name = exp['name']
    
    print(f"--- Analyzing {task_name} ---")
    train_loader, test_loader, input_size, output_size = get_monk_data(monk_id, batch_size)
    X_full = train_loader.dataset.tensors[0].numpy()
    y_full = train_loader.dataset.tensors[1].numpy()
    X_test = test_loader.dataset.tensors[0].numpy()
    y_test = test_loader.dataset.tensors[1].numpy()
    
    # SPLIT
    X_train, X_val, y_train, y_val = train_test_split(X_full, y_full, test_size=0.2, random_state=42)
    
    # Grid
    learning_rates = [0.005, 0.01]
    momentums = [0.9]
    hidden_units_list = [4, 8]
    l2_regs = [0.0] if not is_reg else [0.0001, 0.001]
    epochs_list = [500] 
    
    best_val_acc = 0.0
    best_config = {}
    best_history = {}
    best_weights = None
    
    for lr in learning_rates:
        for mom in momentums:
            for hu in hidden_units_list:
                for l2 in l2_regs:
                    for epochs in epochs_list:
                        
                        keras.backend.clear_session()
                        model = build_monk_model(hu, lr, mom, l2, input_size)
                        
                        # Train
                        hist = model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, 
                                         validation_data=(X_val, y_val), verbose=0)
                        
                        val_acc = hist.history['val_accuracy'][-1]
                        
                        if val_acc > best_val_acc:
                            best_val_acc = val_acc
                            best_config = {'lr':lr, 'momentum':mom, 'hidden_units':hu, 'l2_reg':l2, 'epochs':epochs}
                            best_history = hist.history
                            best_weights = model.get_weights()
                            print(f"  New Best: {best_config}, Val Acc: {val_acc:.4f}")

    print(f"Best Config for {task_name}: {best_config}, Val Acc: {best_val_acc:.4f}")
    
    # Eval Best
    keras.backend.clear_session()
    best_lr = best_config['lr']; best_mom = best_config['momentum']
    best_hu = best_config['hidden_units']; best_l2 = best_config['l2_reg']
    model = build_monk_model(best_hu, best_lr, best_mom, best_l2, input_size)
    model.set_weights(best_weights)
    
    # Evaluate Test
    loss, final_test_acc = model.evaluate(X_test, y_test, verbose=0)
    y_test_pred = model.predict(X_test, verbose=0)
    final_test_mse = np.mean(np.square(y_test.reshape(-1, 1) - y_test_pred))
    
    # Evaluate Train
    y_train_pred = model.predict(X_train, verbose=0)
    final_train_mse = np.mean(np.square(y_train.reshape(-1, 1) - y_train_pred))
    
    print(f"Final Test Accuracy: {final_test_acc:.4f}, Test MSE: {final_test_mse:.4f}")
    print(f"Final Train MSE: {final_train_mse:.4f}")
    
    # Dummy Baseline
    dummy_val = max(1 - y_test.mean(), y_test.mean())
    print(f"Baseline Accuracy: {dummy_val:.4f}")
    
    # Plot
    standard_plot(best_history, f"{task_name} Best Config", f"{task_name.lower().replace('-', '_')}_keras.png", 
                  baseline_value=dummy_val, baseline_label=f"Dummy Acc: {dummy_val:.4f}", 
                  test_score=final_test_acc, test_score_label="Test Acc")
        
    update_json(task_name, "Keras", {
        "test_accuracy": final_test_acc, 
        "test_mse": float(final_test_mse),
        "train_mse": float(final_train_mse),
        "config": best_config
    })


--- Analyzing Monk-3-Reg ---
Parsing MONK-3 data...


/Users/michaelbiggeri/Desktop/Informatica/Projects/Rage_Against_ML/.venv/lib/python3.9/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


  New Best: {'lr': 0.005, 'momentum': 0.9, 'hidden_units': 4, 'l2_reg': 0.0001, 'epochs': 200}, Val Acc: 0.8800
  New Best: {'lr': 0.005, 'momentum': 0.9, 'hidden_units': 8, 'l2_reg': 0.0001, 'epochs': 200}, Val Acc: 0.9200
Best Config for Monk-3-Reg: {'lr': 0.005, 'momentum': 0.9, 'hidden_units': 8, 'l2_reg': 0.0001, 'epochs': 200}, Val Acc: 0.9200
Final Test Accuracy: 0.9444, Test MSE: 0.0958
Final Train MSE: 0.1062
Baseline Accuracy: 0.5278
Saved plot to monk_3_reg_keras.png
Updated all_results.json for Task: Monk-3-Reg, Model: Keras


In [ ]:
# --- MANUAL MONK EXPERIMENT WITH 5-FOLD CV ---
from sklearn.model_selection import KFold
import numpy as np

# Customize these hyperparameters
MANUAL_MONK_ID = 2
MANUAL_IS_REG = False
MANUAL_LR = 0.01
MANUAL_MOMENTUM = 0.9
MANUAL_HIDDEN = 4
MANUAL_L2 = 0.0
MANUAL_EPOCHS = 1000
K_FOLDS = 5

task_name_manual = f"Monk-{MANUAL_MONK_ID}" + ("-Reg" if MANUAL_IS_REG else "")
print(f"--- Running {K_FOLDS}-Fold CV for {task_name_manual} ---")

# Load Data
train_loader, test_loader, input_size, output_size = get_monk_data(MANUAL_MONK_ID, batch_size)

# Extract Arrays
X_full = train_loader.dataset.tensors[0].numpy()
y_full = train_loader.dataset.tensors[1].numpy()
X_test = test_loader.dataset.tensors[0].numpy()
y_test = test_loader.dataset.tensors[1].numpy()

kf = KFold(n_splits=K_FOLDS, shuffle=True, random_state=42)

all_fold_histories = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}
fold_test_accuracies = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_full, y_full)):
    print(f"Starting Fold {fold+1}/{K_FOLDS}...")
    
    X_train_fold, y_train_fold = X_full[train_idx], y_full[train_idx]
    X_val_fold, y_val_fold = X_full[val_idx], y_full[val_idx]
    
    keras.backend.clear_session()
    model = build_monk_model(MANUAL_HIDDEN, MANUAL_LR, MANUAL_MOMENTUM, MANUAL_L2, input_size)
    
    # Custom In-Memory Checkpoint for this fold
    class InMemoryModelCheckpoint(keras.callbacks.Callback):
        def __init__(self):
            super().__init__()
            self.best_weights = None
            self.best_value = -float('inf')
        def on_epoch_end(self, epoch, logs=None):
            current = logs.get('val_accuracy')
            if current is not None and current >= self.best_value:
                self.best_value = current
                self.best_weights = self.model.get_weights()
                
    checkpointer = InMemoryModelCheckpoint()
    
    history = model.fit(
        X_train_fold, y_train_fold,
        epochs=MANUAL_EPOCHS,
        batch_size=batch_size,
        validation_data=(X_val_fold, y_val_fold),
        callbacks=[checkpointer],
        verbose=0
    )
    
    # Store history
    all_fold_histories['loss'].append(history.history['loss'])
    all_fold_histories['accuracy'].append(history.history['accuracy'])
    all_fold_histories['val_loss'].append(history.history['val_loss'])
    all_fold_histories['val_accuracy'].append(history.history['val_accuracy'])
    
    # Test Evaluation for this fold
    if checkpointer.best_weights:
        model.set_weights(checkpointer.best_weights)
    loss, acc = model.evaluate(X_test, y_test, verbose=0)
    fold_test_accuracies.append(acc)

# --- Aggregation and Plotting ---
mean_test_acc = np.mean(fold_test_accuracies)
std_test_acc = np.std(fold_test_accuracies)
print(f"5-Fold CV Result: Test Acc = {mean_test_acc:.4f} (+/- {std_test_acc:.4f})")

# Helper to plot mean+std
def plot_kfold_history(histories, metric, title, filename):
    plt.figure(figsize=(10, 6))
    
    # Extract data (n_folds, n_epochs)
    train_data = np.array(histories[metric])
    # Keras calls it 'val_accuracy' or 'val_loss', mapped correctly in dictionary above
    val_data = np.array(histories[f"val_{metric}"])
    
    epochs = range(1, train_data.shape[1] + 1)
    
    # Mean and Std
    train_mean = np.mean(train_data, axis=0)
    train_std = np.std(train_data, axis=0)
    val_mean = np.mean(val_data, axis=0)
    val_std = np.std(val_data, axis=0)
    
    # Plot Training
    plt.plot(epochs, train_mean, 'b-', label=f'Training {metric.capitalize()}')
    plt.fill_between(epochs, train_mean - train_std, train_mean + train_std, color='blue', alpha=0.15)
    
    # Plot Validation
    plt.plot(epochs, val_mean, 'r-', label=f'Validation {metric.capitalize()}')
    plt.fill_between(epochs, val_mean - val_std, val_mean + val_std, color='red', alpha=0.15)
    
    plt.title(f"{title} ({metric.capitalize()})")
    plt.xlabel('Epochs')
    plt.ylabel(metric.capitalize())
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig(f"monk_results/{filename}")
    plt.show()

dummy_val = max(1 - y_test.mean(), y_test.mean())
print(f"Baseline Accuracy: {dummy_val:.4f}")

plot_kfold_history(all_fold_histories, 'accuracy', f"{task_name_manual} 5-Fold CV", f"manual_kfold_acc_keras.png")
plot_kfold_history(all_fold_histories, 'loss', f"{task_name_manual} 5-Fold CV", f"manual_kfold_loss_keras.png")


# Part 2: ML-CUP

In [ ]:
if keras.backend.backend() != "torch":
    print(f"warning: keras backend is set to {keras.backend.backend()}, restart jupyter kernel!!!!")
    raise RuntimeError()

In [ ]:
keras.backend.backend()

In [ ]:
import json

global config
with open('./config/keras_nn.json') as keras_nn_config:
    config = json.load(keras_nn_config)
    print("config loaded")

In [ ]:
# Root-level fields
batch_size = config["batchSize"]
scaler_enabled = config["scaler"]["enabled"]
scaler_type = config["scaler"]["type"]
input_size = config["inputSize"]
output_size = config["outputSize"]
seed = config["seed"]

In [ ]:
keras.utils.set_random_seed(seed)

In [ ]:
%load_ext tensorboard
# now available at http://localhost:6006/?

In [ ]:
# Dataset initialization
from utils.data_loader import get_ml_cup_data, split_dataloader, cv_fold_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, MaxAbsScaler

def _scaler():
    if not scaler_enabled:
        return None

    if scaler_type == "Standard":
        return StandardScaler()
    elif scaler_type == "MinMax":
        return MinMaxScaler()
    elif scaler_type == "Robust":
        return RobustScaler()
    elif scaler_type == "MaxAbsScaler":
        return MaxAbsScaler()
    else:
        return None

# We use validation_ratio=0.0 and scale_target=False to match the behavior 
# of the old data_loader_2 (which only did a Train/Test split and didn't scale targets).
# We unpack the result to get train_loader and internal_test_loader.
train_loader, _, test_loader, _, _, _= get_ml_cup_data(
    batch_size=batch_size,
    validation_ratio=0.0,
    test_ratio=0.20,  # Standard ratio from old loader
    scaler=_scaler(),
    scale_target=False # Old loader did not scale targets automatically
)

# Note: 'mps_enabled' logic is handled internally: data_loader.py automatically 
# uses float32 tensors (compatible with MPS) and checks for GPU memory pinning.

In [ ]:
train_loader.dataset.X.shape, train_loader.dataset.y.shape

In [ ]:
test_loader.dataset.X.shape, test_loader.dataset.y.shape

In [ ]:
test_loader.dataset.y.shape[1]

In [ ]:
import numpy as np

y_mean = train_loader.dataset.y.mean(axis=0)        # (4,)

y_pred_baseline = np.tile(y_mean, (len(train_loader.dataset.y), 1))

mee_errors = np.linalg.norm(train_loader.dataset.y - y_pred_baseline, axis=1)
mse_errors = np.square(train_loader.dataset.y - y_pred_baseline)

mee_baseline = mee_errors.mean()
mse_baseline = mse_errors.mean()

print("Baseline MEE:", mee_baseline)
print("Baseline MSE:", mse_baseline)

In [ ]:
from losses import MeanEuclidianError

mee = MeanEuclidianError(name="mee", dtype=torch.float32)

In [ ]:
train_dataset = train_loader.dataset
test_dataset = test_loader.dataset

## Grid Search vs Randomized vs Optuna Comparison\n
Before running the full randomized search, we perform a direct comparison between **Grid Search**, **Randomized search** and **Optuna**.\n
We use a expanded budget of **288 trials** for all methods.\n
\n
- **Grid Search**: Exhaustive search over the discrete grid.\n
- **Randomized & Optuna**: Continuous search within the ranges defined by the grid boundaries.\n
\n
**Search Space:**\n
- **Hidden Units**: Fixed at 16\n
- **LR**: Grid [1e-4 .. 5e-2] | Continuous Log-Uniform [1e-4, 5e-2]\n
- **Momentum**: Grid [0.5 .. 0.9] | Continuous Uniform [0.5, 0.9]\n
- **Weight Decay**: Grid [0.0 .. 1e-1] | Continuous Log-Uniform [1e-5, 1e-1]\n
- **Activation**: ['relu', 'tanh']\n

In [ ]:
# --- GRID SEARCH vs OPTUNA COMPARISON ---
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import keras
from keras import layers
import optuna
import time
import random

# 0. Plot Styling (White Background)
plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['grid.color'] = '#e0e0e0'
plt.rcParams['text.color'] = 'black'
plt.rcParams['axes.labelcolor'] = 'black'
plt.rcParams['xtick.color'] = 'black'
plt.rcParams['ytick.color'] = 'black'

# 1. Setup Data
print("--- Starting Grid Search vs Optuna vs Random Search Comparison (288 Trials each) ---")
if 'train_loader' not in globals():
    print("Error: train_loader not found. Run previous cells first.")
else:
    # Extract Numpy Data
    # Ensure we copy to CPU numpy arrays to avoid any device confusion during slicing
    X_train_cmp = train_loader.dataset.X.cpu().numpy()
    y_train_cmp = train_loader.dataset.y.cpu().numpy()
    
    # Manual Split for Comparison (80/20 of the Train set)
    # This mimics the PyTorch validation split
    val_split_idx = int(0.8 * len(X_train_cmp))
    X_t_cmp = X_train_cmp[:val_split_idx]
    y_t_cmp = y_train_cmp[:val_split_idx]
    X_v_cmp = X_train_cmp[val_split_idx:]
    y_v_cmp = y_train_cmp[val_split_idx:]
    
    input_dim = X_train_cmp.shape[1]
    output_dim = y_train_cmp.shape[1]

    # Define Search Space
    grid_space = {
        'lr': [1e-4, 5e-4, 1e-3, 5e-3, 1e-2, 5e-2],
        'momentum': [0.5, 0.7, 0.8, 0.9],
        'weight_decay': [0.0, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1],
        'activation': ['relu', 'tanh']
    }
    fixed_hidden_units = 16
    N_EPOCHS_COMPARE = 20

    def train_and_eval(params):
        # Optimized Keras implementation for Speed
        keras.backend.clear_session() 
        
        reg = keras.regularizers.l2(params['weight_decay']) if params['weight_decay'] > 0 else None
        
        # functional API is slightly faster to build than Sequential sometimes
        inputs = keras.Input(shape=(input_dim,))
        x = layers.Dense(fixed_hidden_units, activation=params['activation'], kernel_regularizer=reg)(inputs)
        x = layers.Dense(fixed_hidden_units, activation=params['activation'], kernel_regularizer=reg)(x)
        outputs = layers.Dense(output_dim)(x)
        model = keras.Model(inputs=inputs, outputs=outputs)
        
        optimizer = keras.optimizers.SGD(learning_rate=params['lr'], momentum=params['momentum'])
        
        # Compile with run_eagerly=False (default) but ensure steps are minimized
        model.compile(optimizer=optimizer, loss='mse')
        
        # Run training - Validation removed for speed parity with PyTorch
        model.fit(
            X_t_cmp, y_t_cmp,
            epochs=N_EPOCHS_COMPARE,
            verbose=0,
            batch_size=32,
            shuffle=True
        )
        
        # Evaluate MEE manually
        # Use batch processing for prediction to ensure speed
        y_pred = model.predict(X_v_cmp, verbose=0)
        
        # Ensure shapes match
        if y_pred.shape != y_v_cmp.shape:
            print(f"Shape mismatch! {y_pred.shape} vs {y_v_cmp.shape}")
            y_pred = y_pred.reshape(y_v_cmp.shape)
            
        diff = y_pred - y_v_cmp
        # Euclidean norm per row, then mean
        mee = np.mean(np.linalg.norm(diff, axis=1))
        return mee

    # --- 2. GRID SEARCH ---
    print("Running Grid Search (Discrete)...")
    grid_results = {'trial': [], 'value': [], 'lr': [], 'momentum': [], 'weight_decay': [], 'activation': []}
    trial_idx = 0
    start_time = time.time()

    for lr in grid_space['lr']:
        for mom in grid_space['momentum']:
            for wd in grid_space['weight_decay']:
                for act in grid_space['activation']:
                    trial_idx += 1
                    params = {'lr': lr, 'momentum': mom, 'weight_decay': wd, 'activation': act}
                    val_mee = train_and_eval(params)
                    
                    grid_results['trial'].append(trial_idx)
                    grid_results['value'].append(val_mee)
                    grid_results['lr'].append(lr)
                    grid_results['momentum'].append(mom)
                    grid_results['weight_decay'].append(wd)
                    grid_results['activation'].append(act)
                    if trial_idx % 20 == 0:
                        print(f"Grid Trial {trial_idx}/288: MEE: {val_mee:.4f}")
    grid_time = time.time() - start_time

    # --- 2.5. RANDOM SEARCH ---
    print("Running Random Search (Continuous)...")
    random_results = {'trial': [], 'value': [], 'lr': [], 'momentum': [], 'weight_decay': [], 'activation': []}
    random.seed(42)
    start_time = time.time()
    TOTAL_TRIALS = 288

    for i in range(TOTAL_TRIALS):
        r_lr = 10 ** random.uniform(np.log10(1e-4), np.log10(5e-2))
        r_mom = random.uniform(0.5, 0.9)
        r_wd = 10 ** random.uniform(np.log10(1e-5), np.log10(1e-1))
        r_act = random.choice(grid_space['activation'])
        
        params = {'lr': r_lr, 'momentum': r_mom, 'weight_decay': r_wd, 'activation': r_act}
        val_mee = train_and_eval(params)
        
        random_results['trial'].append(i + 1)
        random_results['value'].append(val_mee)
        random_results['lr'].append(r_lr)
        random_results['momentum'].append(r_mom)
        random_results['weight_decay'].append(r_wd)
        random_results['activation'].append(r_act)
        if (i+1) % 20 == 0:
            print(f"Random Trial {i+1}/{TOTAL_TRIALS}: MEE: {val_mee:.4f}")
    random_time = time.time() - start_time

    # --- 3. OPTUNA SEARCH ---
    print("Running Optuna Search...")
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    def objective(trial):
        params = {
            'lr': trial.suggest_float('lr', 1e-4, 5e-2, log=True),
            'momentum': trial.suggest_float('momentum', 0.5, 0.9),
            'weight_decay': trial.suggest_float('weight_decay', 1e-5, 1e-1, log=True),
            'activation': trial.suggest_categorical('activation', grid_space['activation'])
        }
        return train_and_eval(params)

    start_time = time.time()
    study_cmp = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
    study_cmp.optimize(objective, n_trials=TOTAL_TRIALS)
    optuna_time = time.time() - start_time

    optuna_results = {'trial': [], 'value': [], 'lr': [], 'momentum': [], 'weight_decay': [], 'activation': []}
    for t in study_cmp.trials:
        optuna_results['trial'].append(t.number + 1)
        optuna_results['value'].append(t.value)
        optuna_results['lr'].append(t.params['lr'])
        optuna_results['momentum'].append(t.params['momentum'])
        optuna_results['weight_decay'].append(t.params['weight_decay'])
        optuna_results['activation'].append(t.params['activation'])

    # --- 4. VISUALIZATION ---
    print(f"Comparison Complete.")
    print(f"Grid Time: {grid_time:.2f}s")
    print(f"Random Time: {random_time:.2f}s")
    print(f"Optuna Time: {optuna_time:.2f}s")

    # Best Value Plot
    grid_best = np.minimum.accumulate(grid_results['value'])
    random_best = np.minimum.accumulate(random_results['value'])
    opt_best = np.minimum.accumulate(optuna_results['value'])

    plt.figure(figsize=(10, 6))
    plt.plot(grid_results['trial'], grid_best, label='Grid Search Best', marker='o', markevery=20)
    plt.plot(random_results['trial'], random_best, label='Random Search Best', marker='s', linestyle='-.', markevery=20)
    plt.plot(optuna_results['trial'], opt_best, label='Optuna Best', marker='x', markevery=20)
    plt.title('Best MEE vs Trial Number')
    plt.xlabel('Trial')
    plt.ylabel('Best MEE')
    plt.legend()
    plt.grid(True)
    plt.show()

    # Scatter Plot
    plt.figure(figsize=(10, 6))
    plt.scatter(grid_results['trial'], grid_results['value'], label='Grid', alpha=0.5, s=15)
    plt.scatter(random_results['trial'], random_results['value'], label='Random', alpha=0.5, marker='s', s=15)
    plt.scatter(optuna_results['trial'], optuna_results['value'], label='Optuna', alpha=0.5, marker='x', s=15)
    plt.title('Trial MEE Distribution')
    plt.xlabel('Trial')
    plt.ylabel('MEE')
    plt.legend()
    plt.grid(True)
    plt.show()


## Randomized Search

In [ ]:
from scipy.stats import loguniform

param_distributions = {
    "reg__model__learning_rate": loguniform(1e-3, 1e-2),
    "reg__model__lambda_1": loguniform(3e-3, 1e-1),
    "reg__model__lambda_2": loguniform(3e-3, 1e-1),
    "reg__model__activation_1": ["relu", "gelu", "leaky_relu"],
    "reg__model__activation_2": ["relu", "gelu", "leaky_relu"],
    "reg__model__dropout_1": loguniform(0.2, 0.5),
    "reg__model__dropout_2": loguniform(0.2, 0.5),
    "reg__model__pca_input_size": [1, 2],
    "reg__pca__n_components": [1, 2],
    "reg__model__seed": [seed],
    "reg__model__output_size": [train_loader.dataset.y.shape[1]],
}

In [ ]:
from executors import RandomizedSearchRegressionExecutor

# using default param_distribution
rs_regression_executor = RandomizedSearchRegressionExecutor(
    train_loader=train_loader,
    units=[(2,2)],
    n_iter=1,
    epochs=1,
    use_PCA=True,
    loss="mean_squared_error",
    baseline=mse_baseline,
    scoring="neg_mean_squared_error",
    seed=seed,
    save_path="keras/models/rs/test",
    verbose=0,
    n_jobs=4
)

In [ ]:
rs_regression_executor.execute()

## Optuna

In [ ]:
import optuna
from executors import OptunaRegressorExecutor

optuna_executor = OptunaRegressorExecutor(
    train_loader=train_loader,
    units=[(12,12)],
    use_pca=True,
    pca_input_size=2,
    n_trials=100,
    epochs=1000,
    seed=seed,
    batch_size=80,
    n_splits=5,
    sampler=optuna.samplers.TPESampler(seed=seed, constant_liar=True, multivariate=True),
    optuna_base_path="keras/models/optuna/regression27-12",
    verbose=0,
    baseline=mse_baseline,
    n_jobs=4
)

In [ ]:
optuna_executor.execute()

In [ ]:
import utils.optuna_utils as uoptuna

study = uoptuna.import_csv("keras/models/optuna/regression27-12/12x12/optuna_results.csv")

In [ ]:
from optuna.visualization import \
    plot_optimization_history, plot_param_importances, plot_parallel_coordinate, plot_contour

In [ ]:
plot_optimization_history(study)

In [ ]:
study.best_params

In [ ]:
plot_param_importances(study)

In [ ]:
plot_parallel_coordinate(study)

In [ ]:
plot_contour(study, params=['dropout_2', 'learning_rate'])

In [ ]:
from utils.plot import plot_optuna_vs_random

plot_optuna_vs_random(
    optuna_csv_path="keras/models/optuna/regression27-12/12x12/optuna_results.csv",
    rs_csv_path="keras/models/rs/regression27-12/12x12/cv_results_df.csv"
                      )

In [ ]:
# --- FINAL CUP EVALUATION ---
print("\n--- Final CUP Evaluation with Best Params ---")
# Assuming 'study' is available from previous cells
best_params = study.best_params
print("Best Params:", best_params)

# Extract params (assuming 12x12 architecture as per notebook default)
unit1 = 12
unit2 = 12

lr = best_params['learning_rate']
l1 = best_params['lambda_1']
act1 = best_params['activation_1']
drop1 = best_params['dropout_1']

l2 = best_params.get('lambda_2', 0.0)
act2 = best_params.get('activation_2', 'relu')
drop2 = best_params.get('dropout_2', 0.0)

meta = {"n_features_in_": input_size, "n_outputs_": output_size}

# Build Model
model = build_model_two_hidden(
    meta, unit1, unit2, seed, lr, drop1, drop2, l1, l2, act1, act2
)

# Train on Full Train Data (Train+Val)
# train_loader in notebook is already Train+Val (validation_ratio=0.0)
X_train = train_loader.dataset.X.numpy()
y_train = train_loader.dataset.y.numpy()
X_test = test_loader.dataset.X.numpy()
y_test = test_loader.dataset.y.numpy()

# Callbacks
callbacks = [
    keras.callbacks.EarlyStopping(monitor='loss', patience=50, restore_best_weights=True)
]

history = model.fit(
    X_train, y_train,
    epochs=1000, # Train longer for final
    batch_size=batch_size,
    verbose=0,
    callbacks=callbacks
)

# Evaluate
# MEE is in metrics (index 1 usually, index 0 is loss)
results = model.evaluate(X_test, y_test, verbose=0)
loss = results[0]
mee_score = results[1]
print(f"Final Internal Test MEE: {mee_score}")

# Save Results
update_json("CUP", "Keras", {"internal_test_MEE": mee_score})

# Dummy Regressor Baseline\n
# MEE calculation manually to avoid dependency issues\n
# Robust Dummy Baseline\n
val_dummy_mee = None\n
try:\n
    # Keras: y_train is likely numpy array available in scope\n
    y_tr_mean = y_train.mean(axis=0)\n
    diff = y_test - y_tr_mean\n
    val_dummy_mee = np.mean(np.sqrt(np.sum(diff**2, axis=1)))\n
    print(f"Dummy MEE: {val_dummy_mee:.4f}")\n
except Exception as e:\n
    print(f"Could not calculate dummy baseline: {e}")\n
standard_plot(history.history, f"CUP Keras Training - MEE: {mee_score:.4f}", "cup_keras.png", baseline_value=val_dummy_mee, baseline_label=f"Dummy MEE: {val_dummy_mee:.4f}" if val_dummy_mee else "Baseline")

# Save the Best Model
import os
os.makedirs("models", exist_ok=True)
model_save_path = "models/best_keras_mlcup.keras"
model.save(model_save_path)
print(f"Best Keras model saved to {model_save_path}")
